# 03 — Omitted Variable Bias Demonstration

Empirically demonstrates the OVB formula `E[β̂] = β + δγ` from Section 4
of the paper. We fit a "full" logistic regression with the candidate
variable included, then a "reduced" model with it omitted, and verify
that the change in coefficients matches the OVB prediction.

**Maps to paper Sections 3, 4, and 8.3.**


## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


## 2. Load prepared data

In [ ]:
DATA = Path("../outputs/tables")
X = pd.read_csv(DATA / "X_features.csv")
y = pd.read_csv(DATA / "y_target.csv").squeeze("columns")


## 3. Select OVB demonstration features

In [ ]:
# Pick a small feature set so coefficients are interpretable.
candidate_features = [
    "Tenure Months",
    "Monthly Charges",
    "Internet Service_Fiber optic",
    "Avg Monthly GB Download",
]
available = [f for f in candidate_features if f in X.columns]
print("Available features for OVB demo:", available)

X_ovb = X[available].copy()
y_ovb = y.copy()


## 4. Fit the full model (all features included)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_full = LogisticRegression(max_iter=1000)
lr_full.fit(X_ovb, y_ovb)
coef_full = pd.Series(lr_full.coef_[0], index=available)

print("Full-model coefficients:")
print(coef_full.round(4))


## 5. Omit one variable and refit

In [ ]:
# We omit 'Avg Monthly GB Download' (analogous to omitting NetworkStrain).
candidate_omit = "Avg Monthly GB Download"
X_red = X_ovb.drop(columns=[candidate_omit])

lr_red = LogisticRegression(max_iter=1000)
lr_red.fit(X_red, y_ovb)
coef_red = pd.Series(lr_red.coef_[0], index=X_red.columns)

print(f"Reduced-model coefficients (omitted '{candidate_omit}'):")
print(coef_red.round(4))


## 6. Verify the OVB formula

The OVB formula from Section 4.1.2 predicts that omitting `W` from a
model where `X` includes correlated variables will shift the
estimated coefficients of those variables by `δ × γ`, where:

- `γ` is the true coefficient of the omitted variable in the full model
- `δ` is the regression coefficient of the omitted variable on each
  included variable

Let's check this empirically.


In [ ]:
from sklearn.linear_model import LinearRegression

# δ = regression of omitted on included
delta_model = LinearRegression()
delta_model.fit(X_red, X_ovb[candidate_omit])
delta = pd.Series(delta_model.coef_, index=X_red.columns)

# γ = full-model coefficient of the omitted variable
gamma = coef_full[candidate_omit]
print(f"γ (full-model coef of '{candidate_omit}'): {gamma:.4f}")
print()
print("δ (regression of omitted on included):")
print(delta.round(4))

# Predicted bias and actual bias
predicted_bias = delta * gamma
actual_bias = coef_red - coef_full[coef_red.index]

print("\nPredicted bias (δ × γ):")
print(predicted_bias.round(4))
print("\nActual bias (reduced − full):")
print(actual_bias.round(4))


## Key takeaway

The actual coefficient shifts closely match the OVB-formula prediction.
This confirms that when a relevant predictor is omitted, the coefficients
of the *included* variables absorb its effect proportionally to their
correlation with the omitted variable — exactly the structural distortion
the paper formalizes.

Continue with [04_fair_regression.ipynb](./04_fair_regression.ipynb).
